# Unified Evaluation + Reporting

Aggregates predictions from multiple runs and produces thesis-ready tables + figures.

Outputs saved to `2_modeling/12_eval_reporting/results/`.


In [1]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve dataset root

def find_kvasir_x1_root() -> Path:
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").exists():
            return (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").resolve()
        if (p / "0_dataset_prep").exists() and (p / "1_dataset_analysis").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not locate Kvasir_VQA_x1 root. Set KVASIR_VQA_X1_ROOT.")

ROOT = find_kvasir_x1_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.metrics import normalize_answer, compute_metrics

MANIFEST = ROOT / "0_dataset_prep" / "out" / "manifest_x1.parquet"
RESULTS_DIR = ROOT / "2_modeling" / "12_eval_reporting" / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIG_DIR = RESULTS_DIR / "figures"

for d in [RESULTS_DIR, TABLES_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Results dir:", RESULTS_DIR)


ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Results dir: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/12_eval_reporting/results


In [2]:
# Load manifest
manifest = pd.read_parquet(MANIFEST)
print("rows:", len(manifest))


rows: 159549


In [3]:
# Config: add/remove models here
EVAL_SPLIT = "test"

MODEL_SPECS = [
    {
        "name": "qwen2_5_vl_zeroshot",
        "pred_path": ROOT / "2_modeling" / "10_modern_vlm" / "results" / "qwen2_5_vl_zeroshot" / "predictions.jsonl",
        "metrics_path": ROOT / "2_modeling" / "10_modern_vlm" / "results" / "qwen2_5_vl_zeroshot" / "metrics.json",
        "format": "jsonl",
        "pred_col": "pred_raw",
        "answer_col": "answer",
    },
    {
        "name": "medgemma_zeroshot",
        "pred_path": ROOT / "2_modeling" / "10_modern_vlm" / "results" / "medgemma_zeroshot" / "predictions.jsonl",
        "metrics_path": ROOT / "2_modeling" / "10_modern_vlm" / "results" / "medgemma_zeroshot" / "metrics.json",
        "format": "jsonl",
        "pred_col": "pred_raw",
        "answer_col": "answer",
    },
    # Optional: previous zero-shot baseline with full metadata
    {
        "name": "llava_zeroshot",
        "pred_path": ROOT / "2_modeling" / "03_vlm_modern_baseline_zeroshot" / "out" / "predictions_test.csv",
        "format": "csv",
        "pred_col": "pred_raw",
        "answer_col": "answer",
    },
]


In [4]:
def load_predictions(spec):
    pred_path = spec.get("pred_path")
    if not pred_path:
        return None
    path = Path(pred_path)
    if not path.exists():
        return None

    if spec["format"] == "jsonl":
        df = pd.read_json(path, lines=True)
    else:
        df = pd.read_csv(path)

    pred_col = spec.get("pred_col", "pred")
    answer_col = spec.get("answer_col", "answer")

    if pred_col not in df.columns:
        print(f"[skip] {spec['name']} missing pred col {pred_col}")
        return None
    if answer_col not in df.columns:
        print(f"[skip] {spec['name']} missing answer col {answer_col}")
        return None

    df = df.copy()
    df["pred_raw"] = df[pred_col].astype(str)
    df["answer"] = df[answer_col].astype(str)

    if "pred_norm" not in df.columns:
        df["pred_norm"] = df["pred_raw"].apply(normalize_answer)

    # Ensure split is present
    if "split" not in df.columns:
        df["split"] = EVAL_SPLIT

    return df


def load_metrics(spec):
    metrics_path = spec.get("metrics_path")
    if not metrics_path:
        return None
    path = Path(metrics_path)
    if not path.exists():
        return None
    with open(path, "r") as f:
        return json.load(f)


In [5]:
def _to_hashable(v):
    if isinstance(v, list):
        return tuple(v)
    try:
        import numpy as _np
        if isinstance(v, _np.ndarray):
            return tuple(v.tolist())
    except Exception:
        pass
    return v


def _unique_manifest_slice(manifest_df: pd.DataFrame, key_cols, meta_cols):
    key_cols = [c for c in key_cols if c in manifest_df.columns]
    meta_cols = [c for c in meta_cols if c in manifest_df.columns]
    if not key_cols:
        return None

    cols = key_cols + meta_cols
    m = manifest_df[cols].copy()

    # Collapse duplicate keys to a single row so merge can be many-to-one.
    if m.duplicated(key_cols).any():
        g = m.groupby(key_cols, dropna=False, sort=False)

        conflict_counts = {}
        for c in meta_cols:
            nunq = g[c].apply(lambda s: s.map(_to_hashable).nunique(dropna=False))
            conflict_counts[c] = int((nunq > 1).sum())

        if any(v > 0 for v in conflict_counts.values()):
            print(f"[warn] duplicate manifest keys on {key_cols}; resolving with first row. conflicts={conflict_counts}")

        agg = {c: "first" for c in meta_cols}
        m = g.agg(agg).reset_index()

    return m


def attach_manifest_fields(pred_df: pd.DataFrame, manifest_df: pd.DataFrame) -> pd.DataFrame:
    meta_cols = ["question_class_list", "complexity", "is_transformed"]
    missing_meta = [c for c in meta_cols if c not in pred_df.columns]

    # Newer result files already include these fields; no merge needed.
    if not missing_meta:
        return pred_df

    key_candidates = [
        ["img_id", "question", "answer", "split"],
        ["img_id", "question", "answer"],
        ["img_id", "question", "split"],
        ["img_id", "question"],
        ["question", "answer", "split"],
        ["question", "answer"],
    ]

    for key_cols in key_candidates:
        if not all(c in pred_df.columns for c in key_cols):
            continue
        if not all(c in manifest_df.columns for c in key_cols):
            continue

        m = _unique_manifest_slice(manifest_df, key_cols, missing_meta)
        if m is None:
            continue

        merged = pred_df.merge(m, on=key_cols, how="left", validate="many_to_one")
        return merged

    return pred_df



def _safe_tokens(text):
    if text is None:
        return []
    s = str(text).strip().lower()
    if not s:
        return []
    return s.split()


def _lcs_len(a_tokens, b_tokens):
    n, m = len(a_tokens), len(b_tokens)
    if n == 0 or m == 0:
        return 0
    dp = [0] * (m + 1)
    for i in range(1, n + 1):
        prev = 0
        ai = a_tokens[i - 1]
        for j in range(1, m + 1):
            tmp = dp[j]
            if ai == b_tokens[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = tmp
    return dp[m]


def _bleu4_corpus(pred_texts, ref_texts):
    from collections import Counter
    import math

    clipped = [0, 0, 0, 0]
    totals = [0, 0, 0, 0]
    c_len = 0
    r_len = 0

    for hyp_text, ref_text in zip(pred_texts, ref_texts):
        hyp = _safe_tokens(hyp_text)
        ref = _safe_tokens(ref_text)
        c_len += len(hyp)
        r_len += len(ref)

        for n in range(1, 5):
            if len(hyp) < n:
                continue
            hyp_ngrams = Counter(tuple(hyp[i:i+n]) for i in range(len(hyp) - n + 1))
            ref_ngrams = Counter(tuple(ref[i:i+n]) for i in range(max(0, len(ref) - n + 1)))
            totals[n - 1] += sum(hyp_ngrams.values())
            for ng, cnt in hyp_ngrams.items():
                clipped[n - 1] += min(cnt, ref_ngrams.get(ng, 0))

    if c_len == 0:
        return 0.0

    # Add-one smoothing to avoid zeroing BLEU on sparse short outputs.
    precisions = [
        (clipped[i] + 1.0) / (totals[i] + 1.0) if totals[i] > 0 else 0.0
        for i in range(4)
    ]
    if any(p <= 0 for p in precisions):
        return 0.0

    bp = 1.0 if c_len > r_len else math.exp(1.0 - (r_len / max(c_len, 1)))
    bleu = bp * math.exp(sum(0.25 * math.log(p) for p in precisions))
    return float(bleu)


def _rouge_l_f1_mean(pred_texts, ref_texts):
    scores = []
    for hyp_text, ref_text in zip(pred_texts, ref_texts):
        hyp = _safe_tokens(hyp_text)
        ref = _safe_tokens(ref_text)
        if not hyp or not ref:
            scores.append(0.0)
            continue
        lcs = _lcs_len(hyp, ref)
        p = lcs / len(hyp)
        r = lcs / len(ref)
        f1 = (2 * p * r / (p + r)) if (p + r) else 0.0
        scores.append(f1)
    return float(np.mean(scores)) if scores else 0.0


def _meteor_mean(pred_texts, ref_texts):
    from collections import defaultdict

    scores = []
    for hyp_text, ref_text in zip(pred_texts, ref_texts):
        hyp = _safe_tokens(hyp_text)
        ref = _safe_tokens(ref_text)
        if not hyp or not ref:
            scores.append(0.0)
            continue

        ref_positions = defaultdict(list)
        for i, tok in enumerate(ref):
            ref_positions[tok].append(i)

        matches = []
        used_ref = set()
        for h_i, tok in enumerate(hyp):
            for r_i in ref_positions.get(tok, []):
                if r_i not in used_ref:
                    used_ref.add(r_i)
                    matches.append((h_i, r_i))
                    break

        m = len(matches)
        if m == 0:
            scores.append(0.0)
            continue

        p = m / len(hyp)
        r = m / len(ref)
        f_mean = (10 * p * r) / (r + 9 * p) if (r + 9 * p) else 0.0

        # Chunk count for METEOR penalty.
        matches.sort()
        chunks = 1
        for k in range(1, m):
            prev_h, prev_r = matches[k - 1]
            cur_h, cur_r = matches[k]
            if not (cur_h == prev_h + 1 and cur_r == prev_r + 1):
                chunks += 1

        penalty = 0.5 * ((chunks / m) ** 3)
        scores.append(f_mean * (1 - penalty))

    return float(np.mean(scores)) if scores else 0.0


def compute_text_gen_metrics(pred_series, ref_series):
    pred_texts = ["" if pd.isna(x) else str(x) for x in pred_series]
    ref_texts = ["" if pd.isna(x) else str(x) for x in ref_series]

    return {
        "bleu": _bleu4_corpus(pred_texts, ref_texts),
        "rouge_l": _rouge_l_f1_mean(pred_texts, ref_texts),
        "meteor": _meteor_mean(pred_texts, ref_texts),
    }


In [6]:
# Load all results (predictions preferred; fall back to metrics.json)
leader_rows = []
by_class_rows = []
by_comp_rows = []
by_trans_rows = []

for spec in MODEL_SPECS:
    df = load_predictions(spec)
    if df is not None:
        df = df[df["split"] == EVAL_SPLIT].reset_index(drop=True)
        df = attach_manifest_fields(df, manifest)
        df["model"] = spec["name"]

        # Overall
        m = compute_metrics(df["pred_norm"], df["answer"])
        m.update(compute_text_gen_metrics(df["pred_raw"], df["answer"]))
        m["model"] = spec["name"]
        leader_rows.append(m)

        # By class
        if "question_class_list" in df.columns:
            exploded = df.copy()
            exploded["question_class_list"] = exploded["question_class_list"].apply(
                lambda x: x if isinstance(x, list) else (x.tolist() if hasattr(x, "tolist") else ([] if pd.isna(x) else [x]))
            )
            exploded = exploded.explode("question_class_list")
            exploded = exploded[exploded["question_class_list"].notna()]
            for qc, g in exploded.groupby("question_class_list"):
                m = compute_metrics(g["pred_norm"], g["answer"])
                m["model"] = spec["name"]
                m["question_class"] = qc
                by_class_rows.append(m)
        else:
            print(f"[warn] {spec['name']} missing question_class_list; skipping class breakdown")

        # By complexity
        if "complexity" in df.columns:
            for comp, g in df.groupby("complexity"):
                m = compute_metrics(g["pred_norm"], g["answer"])
                m["model"] = spec["name"]
                m["complexity"] = comp
                by_comp_rows.append(m)
        else:
            print(f"[warn] {spec['name']} missing complexity; skipping complexity breakdown")

        # By transformed
        if "is_transformed" in df.columns:
            for tval, g in df.groupby("is_transformed"):
                m = compute_metrics(g["pred_norm"], g["answer"])
                m["model"] = spec["name"]
                m["is_transformed"] = bool(tval)
                by_trans_rows.append(m)
        else:
            print(f"[warn] {spec['name']} missing is_transformed; skipping transformed breakdown")

        continue

    metrics = load_metrics(spec)
    if metrics is None:
        print(f"[skip] missing predictions/metrics for {spec['name']}")
        continue

    overall = metrics.get("overall")
    if overall:
        row = dict(overall)
        for k in ["bleu", "rouge_l", "meteor"]:
            row.setdefault(k, np.nan)
        row["model"] = spec["name"]
        leader_rows.append(row)

    for row in metrics.get("by_question_class", []):
        r = dict(row)
        r["model"] = spec["name"]
        by_class_rows.append(r)

    for row in metrics.get("by_complexity", []):
        r = dict(row)
        r["model"] = spec["name"]
        by_comp_rows.append(r)

    for row in metrics.get("by_transformed", []):
        r = dict(row)
        r["model"] = spec["name"]
        by_trans_rows.append(r)

if not leader_rows:
    raise RuntimeError("No predictions or metrics loaded. Check MODEL_SPECS paths.")

leader_df = pd.DataFrame(leader_rows).sort_values("em", ascending=False)
by_class_df = pd.DataFrame(by_class_rows) if by_class_rows else None
by_comp_df = pd.DataFrame(by_comp_rows) if by_comp_rows else None
by_trans_df = pd.DataFrame(by_trans_rows) if by_trans_rows else None

print("Loaded models:", leader_df["model"].unique())


[warn] duplicate manifest keys on ['img_id', 'question', 'answer', 'split']; resolving with first row. conflicts={'question_class_list': 6, 'is_transformed': 0}
Loaded models: ['medgemma_zeroshot' 'qwen2_5_vl_zeroshot' 'llava_zeroshot']


In [7]:
# Compute leaderboard
if "leader_df" not in globals() or leader_df is None or leader_df.empty:
    raise RuntimeError("leader_df is missing/empty. Run the previous load-results cell first.")

leader_df = leader_df.sort_values("em", ascending=False).reset_index(drop=True)

preferred_cols = [
    "model", "em", "token_f1", "anls", "bleu", "rouge_l", "meteor", "count"
]
ordered_cols = [c for c in preferred_cols if c in leader_df.columns] + [
    c for c in leader_df.columns if c not in preferred_cols
]
leader_df = leader_df[ordered_cols]

leader_df.to_csv(TABLES_DIR / "leaderboard.csv", index=False)
print(leader_df)


                 model        em  token_f1      anls      bleu   rouge_l  \
0    medgemma_zeroshot  0.000063  0.213080  0.017498  0.033341  0.158501   
1  qwen2_5_vl_zeroshot  0.000000  0.172788  0.000000  0.017084  0.123496   
2       llava_zeroshot  0.000000  0.212437  0.007032  0.025760  0.163942   

     meteor  count  
0  0.141180  15955  
1  0.187288  15955  
2  0.150097  15955  


In [8]:
# Breakdown by question_class
if by_class_df is not None and not by_class_df.empty:
    by_class_df.to_csv(TABLES_DIR / "breakdown_by_class.csv", index=False)
    print(by_class_df.head())
else:
    print("No question_class results; skipping class breakdown")


    em  token_f1  anls  count                model         question_class
0  0.0  0.161059   0.0   1862  qwen2_5_vl_zeroshot      abnormality_color
1  0.0  0.191574   0.0   1901  qwen2_5_vl_zeroshot   abnormality_location
2  0.0  0.165358   0.0   1823  qwen2_5_vl_zeroshot   abnormality_presence
3  0.0  0.260862   0.0   2319  qwen2_5_vl_zeroshot  box_artifact_presence
4  0.0  0.160905   0.0   2115  qwen2_5_vl_zeroshot          finding_count


In [9]:
# Breakdown by complexity
if by_comp_df is not None and not by_comp_df.empty:
    by_comp_df.to_csv(TABLES_DIR / "breakdown_by_complexity.csv", index=False)
    print(by_comp_df.head())
else:
    print("No complexity results; skipping complexity breakdown")


    em  token_f1      anls  count                model  complexity
0  0.0  0.079875  0.000000   5496  qwen2_5_vl_zeroshot           1
1  0.0  0.171684  0.000000   5251  qwen2_5_vl_zeroshot           2
2  0.0  0.271952  0.000000   5208  qwen2_5_vl_zeroshot           3
3  0.0  0.145365  0.017669   5496    medgemma_zeroshot           1
4  0.0  0.216663  0.021407   5251    medgemma_zeroshot           2


In [10]:
# Original vs transformed
if by_trans_df is not None and not by_trans_df.empty:
    by_trans_df.to_csv(TABLES_DIR / "original_vs_transformed.csv", index=False)
    print(by_trans_df.head())
else:
    print("No transformed results; skipping transformed breakdown")


         em  token_f1      anls  count                model  is_transformed
0  0.000000  0.172788  0.000000  15955  qwen2_5_vl_zeroshot           False
1  0.000063  0.213080  0.017498  15955    medgemma_zeroshot           False
2  0.000000  0.212437  0.007032  15955       llava_zeroshot           False


In [11]:
# Figures

# Leaderboard bar chart (EM)
plt.figure(figsize=(6, 4))
plt.bar(leader_df["model"], leader_df["em"])
plt.title("EM Leaderboard (test)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "leaderboard_em.png", dpi=200)
plt.close()

# Complexity plot if available
if by_comp_df is not None and not by_comp_df.empty:
    for model in by_comp_df["model"].unique():
        sub = by_comp_df[by_comp_df["model"] == model]
        plt.figure(figsize=(5, 3))
        plt.plot(sub["complexity"], sub["em"], marker="o")
        plt.title(f"EM by Complexity — {model}")
        plt.xlabel("Complexity")
        plt.ylabel("EM")
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"em_by_complexity_{model}.png", dpi=200)
        plt.close()

print("Saved figures to", FIG_DIR)


Saved figures to /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/12_eval_reporting/results/figures


## Optional: qualitative error gallery

Add a small curated set (20–40 examples) using the prediction files once they are available.
